# 6 - Basemaps, decoration and projections

Furniture and framing: web-map `tiles` under the data, Natural-Earth `features` (land / rivers / lakes) and
`coastlines`, `labels` / `text` annotation, a map `projection` with a `graticule`, and a Bokeh `theme`.

The web-map layers render in **Web Mercator (EPSG:3857)**, so we reproject the vector data to 3857 first
(`to_crs(3857)`).

**Setup** and data - the German states and a global temperature field.


In [ ]:
import warnings

warnings.filterwarnings("ignore")  # hide backend-mismatch style warnings in the docs

import io
from pathlib import Path

import holoviews as hv
import matplotlib.pyplot as plt
from IPython.display import Image, display

hv.extension("bokeh", "matplotlib")  # Bokeh = interactive (the default); matplotlib only for barbs / projection

# This notebook lives in docs/examples/interactive/guide/, so the bundled sample data at the repo-root
# examples/data/ is four levels up. The kernel's working directory is always the notebook's own folder
# (true for nbval, nbconvert and JupyterLab alike), so one clean relative path is all that is needed.
DATA = Path("../../../../examples/data")

from digitalearth.interactive import InteractiveMap
from pyramids.dataset import Dataset


def show(m, static=False):
    """Return the **live, interactive Bokeh map** - pan, zoom, hover, and the toolbar are all real.

    Run the notebook (Run All / Shift-Enter), or open it in a trusted viewer, to see the maps draw; a saved
    .ipynb only shows them once a kernel executes their JavaScript. A few layers have no Bokeh glyph
    (``projection``, ``streamlines``, wind ``barbs``) and are drawn by matplotlib; for those pass
    ``static=True`` (projected maps switch automatically) to embed a PNG instead.
    """
    if static or getattr(m, "_projection", None) is not None:
        fig = hv.render(m.render(), backend="matplotlib")
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
        plt.close(fig)
        return Image(data=buf.getvalue())
    return m.render()


In [ ]:
import geopandas as gpd

adm1 = gpd.read_file(str(DATA / "germany_adm1.geojson")).to_crs(3857)
adm1_4326 = gpd.read_file(str(DATA / "germany_adm1.geojson"))
adm1_4326["area_km2"] = adm1_4326.to_crs(3035).area / 1e6
ncep = Dataset.read_file(str(DATA / "global" / "ncep_tavg_07.tif"))
print("states:", len(adm1))

## `tiles` - a web-map basemap

`tiles` draws a slippy-map basemap under the data; `list_tile_providers()` lists the names.

In [ ]:
InteractiveMap().list_tile_providers()[:8]

In [ ]:
m = InteractiveMap(crs=3857, title="German states on a Carto basemap")
m.tiles("CartoLight").polygons(adm1, line_color="firebrick", fill_alpha=0.05)
show(m)

## `features` - Natural-Earth reference layers

`features` adds vector reference layers (land, rivers, lakes, borders) from Natural Earth.

In [ ]:
m = InteractiveMap(crs=3857, title="states with land + rivers + lakes")
m.features(land=True, rivers=True, lakes=True).polygons(adm1, line_color="black", fill_alpha=0.0)
show(m)

## `coastlines` - the Natural-Earth coastline

`coastlines` overlays the Natural-Earth coastline - Germany's North Sea and Baltic coasts here.

In [ ]:
m = InteractiveMap(crs=3857, title="German states with coastlines")
m.coastlines().polygons(adm1, line_color="navy", fill_alpha=0.05)
show(m)

## `labels` and `text`

`labels` draws one label per feature from a column; `text` places a single caption at a coordinate.

In [ ]:
m = InteractiveMap(crs=4326, title="labelled states")
m.choropleth(adm1_4326, "area_km2", cmap="viridis").colorbar()
m.labels(adm1_4326.assign(geometry=adm1_4326.representative_point()), "shapeName", text_font_size="7pt")
m.text(10.4, 48.3, "Bavaria", text_font_size="11pt", text_color="white")
show(m)

## `projection` and `graticule`

`projection` reprojects the whole map onto a cartographic projection (`"Robinson"`, `"Mollweide"`, ...) and
`graticule` overlays lat/lon lines; these render via matplotlib, so `show` returns a static image.

In [ ]:
m = InteractiveMap(crs=4326, title="July temperature on a Robinson projection")
m.field(ncep, cmap="RdBu_r").projection("Robinson").graticule()
show(m, static=True)

## `theme` - restyle the whole figure

`theme` sets the Bokeh renderer theme for every layer; pass `None` to restore the default.

In [ ]:
m = InteractiveMap(crs=4326, title="night_sky theme")
m.theme("night_sky").choropleth(adm1_4326, "area_km2", cmap="viridis").colorbar()
show(m)